# Qwen3-VL-Embedding-8B でメッセージのembeddingを生成

このノートブックでは、traQのメッセージからテキストと画像を抽出し、Qwen3-VL-Embedding-8Bモデルでembeddingを生成します。


In [1]:
# 必要なパッケージのインストール
%pip install transformers accelerate torch pillow requests python-dotenv -q

import torch
import gc

# メモリクリア
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: Tesla T4
GPU Memory: 14.74 GB


In [2]:
# messages.pyのコードを定義
from typing import Iterable, Optional
import requests
import os

messageIds = [
    "019b2a64-eebc-7815-84c5-e384fabc495c", #ictトラブルシューティング予選結果
    "019a48ff-b175-783c-93fc-b4ec6731dde5", #TechBookの販売記録
    "019bb6b8-5c57-7d1f-8b93-297c8e6c7827", #ポータブルモニターを購入
    "019bb5c0-5409-7b3b-ba9b-07449030364a", #大学の課題の締め切り
    "019bd043-d86c-79b8-9cf1-11351a482267", #pcの空き容量の画像
    "0197c4ef-8be2-792e-922b-c498b6948775", #traqの9点リーダーのアイコン
    "01963951-0f55-75b5-a53b-07f467ad9051", #sysad体験会
    "0195be37-dea6-7902-8d04-e46185873108", #githubへの招待
]

def _get_session() -> requests.Session:
    r_session = os.getenv("r_session")
    if not r_session:
        raise RuntimeError("r_session が見つかりません（環境変数を設定してください）")
    
    session = requests.Session()
    session.cookies.set("r_session", r_session)
    return session

def get_messages(target_message_ids: Optional[Iterable[str]] = None):
    base_url = "https://q.trap.jp/api/v3"
    ids = list(target_message_ids) if target_message_ids is not None else messageIds
    
    result = []
    
    with _get_session() as session:
        for message_id in ids:
            url = f"{base_url}/messages/{message_id}"
            
            try:
                response = session.get(url)
                response.raise_for_status()
                result.append(response.json())
            except requests.exceptions.RequestException as e:
                print(f"Error fetching message {message_id}: {e}")
    
    return result

def download_file(file_id: str, base_url: str = "https://q.trap.jp/api/v3") -> bytes:
    with _get_session() as session:
        url = f"{base_url}/files/{file_id}/raw"
        response = session.get(url)
        response.raise_for_status()
        return response.content

print("messages.pyの関数を定義しました")


messages.pyの関数を定義しました


In [3]:
# traQ APIのセッション情報を設定
# Colabのシークレット機能を使うか、直接設定してください
import os

# 方法1: 直接設定（セキュリティ上推奨されませんが、テスト用）
# os.environ["r_session"] = "your_session_token_here"

# 方法2: Colabのシークレットを使う場合（推奨）
# from google.colab import userdata
# os.environ["r_session"] = userdata.get('r_session')

# 方法3: 入力で設定
r_session = input("r_sessionを入力してください: ").strip()
if r_session:
    os.environ["r_session"] = r_session
    print("r_sessionを設定しました")
else:
    print("警告: r_sessionが設定されていません")


警告: r_sessionが設定されていません


In [4]:
# main.pyのコードを定義
import base64
import io
import json
import re
from typing import Any, Dict, List, Optional, Tuple

from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoProcessor

MODEL_NAME = "Qwen/Qwen3-VL-Embedding-8B"
BASE_URL = "https://q.trap.jp/api/v3"

def last_token_pool(
    last_hidden_states: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """Qwen3-Embeddingで推奨される最後のトークンのhidden stateを取得"""
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[
            torch.arange(batch_size, device=last_hidden_states.device),
            sequence_lengths,
        ]

def extract_text(message: Dict[str, Any]) -> str:
    for key in ("content", "text", "body", "message"):
        value = message.get(key)
        if isinstance(value, str) and value.strip():
            return value
    return ""

def extract_image_file_ids(message: Dict[str, Any]) -> List[str]:
    file_ids: List[str] = []
    
    files = message.get("files")
    if isinstance(files, list):
        for item in files:
            if not isinstance(item, dict):
                continue
            file_id = item.get("id") or item.get("fileId")
            mime = item.get("mime") or item.get("mimeType") or item.get("type")
            if file_id and isinstance(mime, str) and mime.startswith("image/"):
                file_ids.append(file_id)
    
    content = extract_text(message)
    if content:
        patterns = [
            r"/files/([0-9a-fA-F-]{16,})",
            r"https?://q\.trap\.jp/[^\s]*/files/([0-9a-fA-F-]{16,})",
            r"https?://q\.trap\.jp/api/v3/files/([0-9a-fA-F-]{16,})",
        ]
        for pattern in patterns:
            for match in re.findall(pattern, content):
                file_ids.append(match)
    
    return list(dict.fromkeys(file_ids))

def extract_inline_images(message: Dict[str, Any]) -> List[Image.Image]:
    images: List[Image.Image] = []
    
    inline = message.get("images") or message.get("image")
    if isinstance(inline, list):
        candidates = inline
    else:
        candidates = [inline] if inline is not None else []
    
    for item in candidates:
        if isinstance(item, dict):
            data = item.get("data") or item.get("base64")
        elif isinstance(item, str):
            data = item
        else:
            data = None
        
        if not isinstance(data, str):
            continue
        
        if data.startswith("data:image"):
            try:
                header, b64 = data.split(",", 1)
                image_bytes = io.BytesIO(base64.b64decode(b64))
                images.append(Image.open(image_bytes).convert("RGB"))
            except Exception:
                continue
    
    return images

def load_images_from_message(message: Dict[str, Any]) -> List[Image.Image]:
    images: List[Image.Image] = []
    images.extend(extract_inline_images(message))
    
    for file_id in extract_image_file_ids(message):
        try:
            content = download_file(file_id, base_url=BASE_URL)
            images.append(Image.open(io.BytesIO(content)).convert("RGB"))
        except Exception as exc:
            print(f"画像の取得に失敗: {file_id} -> {exc}")
    
    return images

print("main.pyの関数を定義しました")


main.pyの関数を定義しました


In [5]:
# メモリ最適化版のload_model関数
def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    
    print(f"デバイス: {device}, dtype: {dtype}")
    
    # GPUメモリ情報を表示
    if device == "cuda":
        print(f"利用可能なGPUメモリ: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
        print(f"現在の使用メモリ: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    print(f"モデルをロード中: {MODEL_NAME}")
    
    # Qwen3-VL-Embedding-8Bのロード（メモリ最適化版）
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map="auto" if device == "cuda" else None,
        low_cpu_mem_usage=True,  # CPUメモリ使用量を削減
    )
    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
    )
    model.eval()
    
    # 推論モードで勾配計算を無効化（メモリ節約）
    for param in model.parameters():
        param.requires_grad = False
    
    if device == "cpu":
        model.to(device)
    
    print("モデルのロード完了")
    if device == "cuda":
        print(f"ロード後の使用メモリ: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    return model, processor, device


In [6]:
# encode_batch関数（メモリ最適化版）
def encode_batch(
    model,
    processor,
    texts: List[str],
    images: Optional[List[Image.Image]],
    device: str,
) -> torch.Tensor:
    """
    テキストと画像をembeddingに変換する。
    Qwen3-Embeddingでは最後のトークンのhidden stateを使用（last token pooling）。
    """
    # モデルに組み込みのencodeメソッドがあれば使用
    if hasattr(model, "get_embedding"):
        try:
            return model.get_embedding(text=texts, images=images)
        except Exception as e:
            print(f"get_embeddingメソッドでエラー: {e}")
    
    # 画像がある場合の処理
    if images:
        processed_texts = []
        for text in texts:
            if "<image>" not in text and "<|image|>" not in text:
                processed_texts.append(f"<|vision_start|><|image_pad|><|vision_end|>{text}")
            else:
                processed_texts.append(text)
        
        inputs = processor(
            text=processed_texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )
    else:
        inputs = processor(
            text=texts,
            return_tensors="pt",
            padding=True,
        )
    
    # デバイスに移動
    inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Qwen3-Embedding標準: last token pooling
    if hasattr(outputs, "last_hidden_state"):
        hidden_states = outputs.last_hidden_state
        attention_mask = inputs.get("attention_mask")
        
        if attention_mask is not None:
            embeddings = last_token_pool(hidden_states, attention_mask)
        else:
            embeddings = hidden_states[:, -1]
        
        # L2正規化
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        # メモリ節約: 不要なテンソルを削除
        del hidden_states, outputs
        if device == "cuda":
            torch.cuda.empty_cache()
        
        return embeddings
    
    # フォールバック
    if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
        result = F.normalize(outputs.pooler_output, p=2, dim=1)
        del outputs
        if device == "cuda":
            torch.cuda.empty_cache()
        return result
    
    result = F.normalize(outputs[0][:, -1], p=2, dim=1)
    del outputs
    if device == "cuda":
        torch.cuda.empty_cache()
    return result


In [7]:
# embed_message関数
def embed_message(
    model,
    processor,
    message: Dict[str, Any],
    device: str,
) -> Tuple[List[float], int]:
    text = extract_text(message)
    images = load_images_from_message(message)
    
    if images:
        texts = [text] * len(images)
        emb = encode_batch(model, processor, texts, images, device)
        emb = emb.mean(dim=0)
    else:
        emb = encode_batch(model, processor, [text], None, device)[0]
    
    return emb.float().cpu().tolist(), len(images)


## モデルのロード

最初にモデルをロードします。これには数分かかる場合があります。


In [ ]:
# モデルをロード
model, processor, device = load_model()


デバイス: cuda, dtype: torch.float16
利用可能なGPUメモリ: 14.74 GB
現在の使用メモリ: 0.00 GB
モデルをロード中: Qwen/Qwen3-VL-Embedding-8B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

## メッセージの取得とembedding生成

メッセージを取得して、それぞれのembeddingを生成します。


In [ ]:
# メッセージを取得
messages = get_messages()
print(f"取得したメッセージ数: {len(messages)}")


In [ ]:
# embeddingを生成
import os

os.makedirs("out", exist_ok=True)

outputs = []
for i, message in enumerate(messages):
    message_id = message.get("id") or message.get("messageId") or "unknown"
    print(f"\n[{i+1}/{len(messages)}] 処理中: {message_id}")
    
    try:
        embedding, image_count = embed_message(model, processor, message, device)
        outputs.append(
            {
                "messageId": message_id,
                "text": extract_text(message),
                "imageCount": image_count,
                "embedding": embedding,
            }
        )
        print(f"✓ 完了: 埋め込み次元={len(embedding)} 画像={image_count}")
        
        # メモリクリア（定期的に）
        if (i + 1) % 3 == 0 and device == "cuda":
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"✗ エラー: {e}")
        import traceback
        traceback.print_exc()

print(f"\n処理完了: {len(outputs)}件のembeddingを生成しました")


In [ ]:
# 結果を保存
with open("out/embeddings.json", "w", encoding="utf-8") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)

print("結果を out/embeddings.json に保存しました")

# 結果のサマリーを表示
if outputs:
    print(f"\n生成されたembeddingの次元: {len(outputs[0]['embedding'])}")
    print(f"画像を含むメッセージ数: {sum(1 for o in outputs if o['imageCount'] > 0)}")
